## Project Plan: Company Report Generator

This project aims to create a web application that scrapes company websites and generates comprehensive reports. Here's a breakdown of the steps we'll follow:

### Step 1: Project Setup and Environment
*   **Set up the development environment**: This will involve installing necessary libraries for web scraping, web development, and data processing.
*   **Define project structure**: Organize files and folders for the scraper, web application, and report templates.

### Step 2: Web Scraping Component
*   **Choose a scraping library**: We'll select a suitable Python library (e.g., Beautiful Soup, Scrapy, Playwright, or Selenium) based on the complexity of the websites to be scraped.
*   **Basic URL fetching**: Implement a function to fetch the content of a given URL.
*   **Text extraction**: Develop logic to extract all visible text content from the webpage.
*   **Image extraction**: Implement methods to find and download images, including their URLs and potentially alt text.
*   **Other data extraction**: Identify and extract other relevant data points like links, meta descriptions, contact information, etc., based on typical company website structures.

### Step 3: Data Processing and Storage
*   **Data cleaning**: Process the extracted text and other data to remove irrelevant content and standardize formats.
*   **Data structuring**: Organize the extracted information into a structured format (e.g., JSON, Python dictionaries) for easy report generation.
*   **Local storage (optional)**: Consider storing scraped data temporarily or persistently for analysis or report regeneration.

### Step 4: Report Generation
*   **Report template design**: Create a template for the company report (e.g., using Markdown, HTML, or a PDF generation library).
*   **Populate report**: Dynamically insert the scraped and processed data into the report template.
*   **Output format**: Decide on the output format for the report (e.g., text file, PDF, HTML file).

### Step 5: Web Interface
*   **Choose a web framework**: Select a lightweight Python web framework (e.g., Flask or Streamlit) for the user interface.
*   **Input form**: Create a web form where users can enter a company URL.
*   **Trigger scraping and report generation**: Integrate the scraping and report generation logic with the web application.
*   **Display report**: Present the generated report to the user in the web interface or provide a download link.

***

Ready to start with **Step 1: Project Setup and Environment**?

## Resources
Hosted on GitHub.

In [ ]:
import sys

# Install Streamlit if it's not already installed
!{sys.executable} -m pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 38.5 MB/s eta 0:00:00


Now that Streamlit is installed, let's create a basic Streamlit app. This app will have a title, a text input field for the company URL, and a button to submit the URL. To run this Streamlit app, save the code below as a Python file (e.g., `app.py`) and then run `streamlit run app.py` from your terminal. For Colab, we'll use `%%writefile` to create the file and then run it.

In [ ]:
%%writefile app.py

import streamlit as st

st.set_page_config(layout="wide")

st.title('Company Report Generator')
st.write('Enter a company URL below to generate a report.')

company_url = st.text_input('Company Website URL', placeholder='e.g., https://www.example.com')

if st.button('Generate Report'):
    if company_url:
        st.write(f'Scraping and generating report for: {company_url}')
        # Placeholder for scraping and report generation logic
        st.success('Report generation initiated (placeholder).')
    else:
        st.error('Please enter a URL.')

Writing app.py


To run the Streamlit app directly within Colab, you'll need to use `!streamlit run` and expose the port. We'll use `nohup` and `&` to run it in the background and output to a file, so it doesn't block the Colab cell execution. You'll then get a public URL to access the app.

In [ ]:
import subprocess
import time
import os
import re
from google.colab import userdata # Import userdata

# Kill any running Streamlit or cloudflared processes from previous runs
!kill $(lsof -t -i:8501) 2>/dev/null || true # Streamlit default port
!pkill -f cloudflared 2>/dev/null || true # cloudflared process

# 1. Install cloudflared
# Download and install cloudflared if not already present
if not os.path.exists('./cloudflared'):
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !mv cloudflared-linux-amd64 cloudflared
    !chmod +x cloudflared

# 2. Run cloudflared tunnel in the background
print("Launching cloudflared tunnel...")
# Start cloudflared in a subprocess, redirecting output to a file
# This makes it easier to capture the public URL
cloudflared_process = subprocess.Popen([
    './cloudflared', 'tunnel', '--url', 'http://localhost:8501'
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Give cloudflared some time to start and print the URL
time.sleep(5)

# Read cloudflared's output to find the public URL
public_url = None
output_lines = []
time_start = time.time()
while time.time() - time_start < 30: # Wait up to 30 seconds for the URL
    line = cloudflared_process.stdout.readline()
    if not line: # EOF or process died
        break
    output_lines.append(line)
    print(f"[Cloudflared Output] {line.strip()}") # Print output for debugging
    match = re.search(r'https://[-a-zA-Z0-9@:%._\+~#=]{1,256}\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

if public_url:
    print(f"\nCloudflared tunnel established. Public URL: {public_url}\n")
else:
    print("\nCould not find cloudflared public URL. Please check the output above for errors.\n")
    print("Cloudflared output:")
    print(''.join(output_lines))

# 3. Run Streamlit in the background
print("Launching Streamlit app (it might take a moment to appear at the public URL)...")

# Retrieve API key from Colab secrets and pass it as an environment variable
colab_api_key = userdata.get('GOOGLE_API_KEY')
if not colab_api_key:
    print("Warning: GOOGLE_API_KEY not found in Colab secrets. Report generation may fail.")

# Prepare environment variables for Streamlit subprocess
streamlit_env = os.environ.copy()
if colab_api_key:
    streamlit_env['GOOGLE_API_KEY'] = colab_api_key # Pass the key as an environment variable

streamlit_process = subprocess.Popen([
    'streamlit', 'run', 'app.py', '--server.port', '8501',
    '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false'
], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, env=streamlit_env) # Pass env to Popen

# Print the Streamlit server URL for reference (it's local, but cloudflared forwards to it)
print("Streamlit local server is running on http://localhost:8501")

if public_url:
    print(f"\nYour Streamlit app should now be accessible at: {public_url}\n")
    print("**Important:** Keep this cell running to keep the Streamlit app and tunnel active.")
else:
    print("\nStreamlit app is running, but the public URL could not be determined. Check Cloudflared output for issues.")

# To prevent the cell from finishing and killing the processes, we can keep it alive
# However, it's better to let the user manually kill it or restart the runtime
# for a clean shutdown.
# For now, we'll just let the subprocesses run in the background. The user can stop the cell if needed.

^C
Launching cloudflared tunnel...


In [ ]:
import sys

# Install web scraping libraries
!{sys.executable} -m pip install requests beautifulsoup4

Now that the necessary libraries are installed, let's update `app.py` to include the scraping functions and use them when the 'Generate Report' button is clicked. We'll define functions to fetch the page content, extract all text, and find image URLs.

In [ ]:
%%writefile app.py

import streamlit as st
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

st.set_page_config(layout="wide")

st.title('Company Report Generator')
st.write('Enter a company URL below to generate a report.')

def get_all_text(soup):
    """Extracts all visible text from a BeautifulSoup object."""
    texts = soup.find_all(text=True)
    visible_texts = filter(tag_visible, texts)
    return u" ".join(t.strip() for t in visible_texts)

def tag_visible(element):
    """Helper function to filter out invisible HTML elements."""
    if element.parent.name in ['style', 'script', 'head', 'title', 'meta', '[document]']:
        return False
    if isinstance(element, Comment):
        return False
    return True

def get_image_urls(soup, base_url):
    """Extracts all image URLs from a BeautifulSoup object."""
    img_urls = []
    for img in soup.find_all('img'):
        src = img.get('src')
        if src:
            # Resolve relative URLs
            abs_url = urljoin(base_url, src)
            img_urls.append(abs_url)
    return img_urls

def scrape_website(url):
    """Fetches a URL, extracts text and image URLs."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        soup = BeautifulSoup(response.text, 'html.parser')

        text_content = get_all_text(soup)
        image_urls = get_image_urls(soup, url)

        return {
            'url': url,
            'text_content': text_content,
            'image_urls': image_urls
        }
    except requests.exceptions.RequestException as e:
        st.error(f"Error fetching URL: {e}")
        return None
    except Exception as e:
        st.error(f"An unexpected error occurred during scraping: {e}")
        return None

# --- Streamlit UI ---

company_url = st.text_input('Company Website URL', placeholder='e.g., https://www.example.com')

from bs4 import Comment # Import Comment class here, needs to be after initial imports

if st.button('Generate Report'):
    if company_url:
        st.info(f'Scraping and generating report for: {company_url}')

        # Perform scraping
        scraped_data = scrape_website(company_url)

        if scraped_data:
            st.success('Scraping complete! Displaying extracted data:')

            st.subheader('Extracted Text Content')
            st.text_area('Text', scraped_data['text_content'], height=300)

            st.subheader('Extracted Image URLs')
            if scraped_data['image_urls']:
                for img_url in scraped_data['image_urls']:
                    st.markdown(f"- {img_url}")
                    # Optional: Display images if you want
                    # try:
                    #     st.image(img_url, width=200)
                    # except:
                    #     st.write(f"Could not display image: {img_url}")
            else:
                st.write("No image URLs found.")

            # You can add more sections here for other data types

        else:
            st.error('Failed to scrape the website.')
    else:
        st.error('Please enter a URL.')

Overwriting app.py


In [ ]:
import sys

# Install the Google Generative AI library
!{sys.executable} -m pip install google-generativeai

To use the Gemini API, you'll need an API key. If you don't already have one, create a key in Google AI Studio. In Colab, add the key to the secrets manager under the "🔑" icon in the left panel. Give it the name `GOOGLE_API_KEY`.

Now, let's modify `app.py` to integrate the Gemini API. This will involve:
1.  **Importing necessary libraries** (`google.generativeai` and `userdata` from `google.colab`).
2.  **Configuring the Gemini API** with your `GOOGLE_API_KEY`.
3.  **Initializing the Gemini model**.
4.  **Creating a function** to prompt Gemini with the scraped text.
5.  **Updating the Streamlit UI** to call this function and display the generated report.

In [ ]:
%%writefile app.py

import streamlit as st
import requests
from bs4 import BeautifulSoup, Comment
from urllib.parse import urljoin, urlparse
import google.generativeai as genai
import os
import re # New import for regular expressions
from datetime import datetime, timedelta

# Max text length to send to Gemini to prevent API limits or excessively long reports for very large sites
MAX_TEXT_LENGTH = 15000
MAX_NEWS_ARTICLES = 5 # Limit the number of news articles to display and send to Gemini

# Retrieve GOOGLE_API_KEY from environment variables
GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')

if not GOOGLE_API_KEY:
    st.error("GOOGLE_API_KEY not found. Please ensure it's set as an environment variable before launching the app.")
    st.stop()

genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel('gemini-flash-latest') # Changed to gemini-flash-latest for consistency

st.set_page_config(layout="wide")

st.title('Company Report Generator')
st.write('Enter a company URL and name below to generate a report.')

def get_all_text(soup):
    """Extracts all visible text from a BeautifulSoup object."""
    texts = soup.find_all(text=True)
    visible_texts = filter(tag_visible, texts)
    return u" ".join(t.strip() for t in visible_texts if t.strip())

def tag_visible(element):
    """Helper function to filter out invisible HTML elements."""
    if element.parent.name in ['style', 'script', 'head', 'title', 'meta', '[document]']:
        return False
    if isinstance(element, Comment):
        return False
    return True

def extract_contact_info(text_content, html_content):
    """Extracts email addresses, phone numbers, and LinkedIn URLs from text and HTML content."""
    emails = set()
    phone_numbers = set()
    linkedin_profiles = set()

    # Email pattern (improved for robustness)
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
    emails.update(re.findall(email_pattern, text_content))

    # Phone number pattern (more comprehensive, allowing various formats and optional country codes)
    phone_pattern = r'\b(?:\+?\d{1,3}[-.\s]?)?\(?\d{2,5}\)?[-.\s]?\d{2,5}[-.\s]?\d{4}\b'
    phone_numbers.update(re.findall(phone_pattern, text_content))

    # LinkedIn profile URL pattern (corrected to properly capture 'www.' or no subdomain)
    linkedin_pattern = r'https?:\/\/(?:www\.)?linkedin\.com\/in\/[A-z0-9_-]+\/?'
    linkedin_profiles.update(re.findall(linkedin_pattern, html_content))

    return list(emails), list(phone_numbers), list(linkedin_profiles)

def search_google_news(company_name):
    """Searches Google News for recent mentions of the company."""
    st.info(f"Searching Google News for '{company_name}'...")
    news_mentions = []
    # Search for news in the last 30 days
    thirty_days_ago = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')
    search_query = f'\"{\ncompany_name}\" after:{thirty_days_ago}'
    google_news_url = f"https://news.google.com/search?q={search_query}&hl=en-US&gl=US&ceid=US:en"

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    try:
        response = requests.get(google_news_url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # Google News HTML structure can change. This is a common pattern for article links.
        # Look for elements that represent news articles
        articles = soup.find_all('a', class_ = re.compile(r'VDXfz')) # This class name is heuristic and might change

        for article in articles[:MAX_NEWS_ARTICLES]:
            title = article.text.strip()
            href = article.get('href')
            if title and href and not href.startswith('#'): # Exclude internal links
                full_url = urljoin(google_news_url, href)
                news_mentions.append({'title': title, 'url': full_url})

    except requests.exceptions.RequestException as e:
        st.warning(f"Could not fetch news from Google News: {e}. News scraping may be unreliable due to anti-bot measures.")
    except Exception as e:
        st.warning(f"An error occurred during news scraping: {e}. News scraping may be unreliable.")

    if not news_mentions:
        st.info("No recent news mentions found or scraping failed. This feature can be unreliable.")

    return news_mentions

def scrape_website(url, company_name):
    """Fetches a URL, extracts text, contact info, and news mentions."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        soup = BeautifulSoup(response.text, 'html.parser')

        text_content = get_all_text(soup)

        # Apply text length limiter
        if len(text_content) > MAX_TEXT_LENGTH:
            text_content = text_content[:MAX_TEXT_LENGTH] + "\n... (text truncated)"
            st.warning(f"Text content truncated to {MAX_TEXT_LENGTH} characters for processing by Gemini.")

        emails, phone_numbers, linkedin_profiles = extract_contact_info(text_content, response.text)
        news_mentions = search_google_news(company_name) if company_name else []

        return {
            'url': url,
            'text_content': text_content,
            'contact_emails': emails,
            'contact_phone_numbers': phone_numbers,
            'contact_linkedin_profiles': linkedin_profiles,
            'news_mentions': news_mentions
        }
    except requests.exceptions.RequestException as e:
        st.error(f"Error fetching URL: {e}")
        return None
    except Exception as e:
        st.error(f"An unexpected error occurred during scraping: {e}")
        return None

def generate_company_report(text_content, contact_emails, contact_phone_numbers, contact_linkedin_profiles, news_mentions):
    """Generates a comprehensive report using Gemini API."""
    news_section = ""
    if news_mentions:
        news_items = "\n".join([f"- {item['title']} (Source: {item['url']})" for item in news_mentions])
        news_section = f"\n\nRecent News Mentions:\n{news_items}\n\nSummarize these news mentions and cite the sources clearly."

    prompt = f"""Generate a comprehensive company report based on the following extracted website content.
    Focus on key information such as:
    - What the company does, its main products/services, and target audience.
    - Identify and summarize information about the leadership team and other key contact persons, if mentioned in the text.
    - Summarize any extracted contact information like email addresses, phone numbers, and LinkedIn profiles.
    {news_section}

    Extracted Text:\n{text_content}\n\n
    Extracted Contact Emails (if any):\n{', '.join(contact_emails) if contact_emails else 'No contact emails found.'}\n\n
    Extracted Phone Numbers (if any):\n{', '.join(contact_phone_numbers) if contact_phone_numbers else 'No phone numbers found.'}\n\n
    Extracted LinkedIn Profiles (if any):\n{', '.join(contact_linkedin_profiles) if contact_linkedin_profiles else 'No LinkedIn profiles found.'}\n\n
    Provide a well-structured report. """

    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        st.error(f"Error generating report with Gemini API: {e}")
        return "Could not generate report."

# --- Streamlit UI ---

company_url = st.text_input('Company Website URL', placeholder='e.g., https://www.example.com')
company_name = st.text_input('Company Name for News Search (e.g., Google, Microsoft)', placeholder='e.g., Example Corp')

if st.button('Generate Report'):
    if company_url:
        st.info(f'Scraping {company_url} and generating report...')

        # Perform scraping
        scraped_data = scrape_website(company_url, company_name)

        if scraped_data:
            st.success('Scraping complete! Generating report...')

            # Generate report using Gemini API
            report = generate_company_report(
                scraped_data['text_content'],
                scraped_data['contact_emails'],
                scraped_data['contact_phone_numbers'],
                scraped_data['contact_linkedin_profiles'],
                scraped_data['news_mentions']
            )

            st.subheader('Company Report')
            st.markdown(report)

            st.subheader('Extracted Text Content (for reference)')
            st.text_area('Text', scraped_data['text_content'], height=200)

            # New UI section for contact emails
            st.subheader('Extracted Contact Emails (for reference)')
            if scraped_data['contact_emails']:
                for email in scraped_data['contact_emails']:
                    st.markdown(f"- {email}")
            else:
                st.write("No contact emails found on the main page.")

            # New UI section for contact phone numbers
            st.subheader('Extracted Phone Numbers (for reference)')
            if scraped_data['contact_phone_numbers']:
                for phone in scraped_data['contact_phone_numbers']:
                    st.markdown(f"- {phone}")
            else:
                st.write("No phone numbers found on the main page.")

            # New UI section for contact LinkedIn profiles
            st.subheader('Extracted LinkedIn Profiles (for reference)')
            if scraped_data['contact_linkedin_profiles']:
                for linkedin_url in scraped_data['contact_linkedin_profiles']:
                    st.markdown(f"- {linkedin_url}")
            else:
                st.write("No LinkedIn profiles found on the main page.")

            # New UI section for news mentions
            st.subheader('Recent News Mentions (for reference)')
            if scraped_data['news_mentions']:
                for news_item in scraped_data['news_mentions']:
                    st.markdown(f"- [{news_item['title']}]({news_item['url']})")
            else:
                st.write("No recent news mentions found.")

        else:
            st.error('Failed to scrape the website.')
    else:
        st.error('Please enter a URL.')

Overwriting app.py


In [ ]:
import google.generativeai as genai
import os
from google.colab import userdata

# Retrieve API key from Colab secrets
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

if GOOGLE_API_KEY:
    genai.configure(api_key=GOOGLE_API_KEY)
    print("API key configured successfully.")
else:
    print("GOOGLE_API_KEY not found in Colab secrets. Please ensure it's set.")
    print("Cannot list models without an API key.")

if GOOGLE_API_KEY:
    print("\nAvailable Gemini Models (for generateContent method):")
    # The FutureWarning recommends switching to google.genai, but for now we'll continue with google.generativeai
    # for consistency with app.py. This should be addressed in future refactoring.
    for m in genai.list_models():
        if 'generateContent' in m.supported_generation_methods:
            print(f"- {m.name}")
else:
    print("Skipping model listing as API key is not configured.")

After executing the cell above, `app.py` will be updated to include the Gemini API integration. You will need to **re-run cell `e7fe2ae3`** to restart the Streamlit application with the updated `app.py` file. Then, revisit the public URL to see the changes.

Make sure your `GOOGLE_API_KEY` is set in Colab secrets. Now, when you enter a URL and click 'Generate Report', the app will scrape the website and then use Gemini to provide a comprehensive report.

After executing the cell above, `app.py` will be updated with the scraping logic, including the news mentions feature. You will need to **re-run cell `e7fe2ae3`** to restart the Streamlit application with the updated `app.py` file. Then, revisit the public URL to see the changes.

Now, when you enter a URL and click 'Generate Report', the app will scrape the website, search for news, and display the extracted text, contact information, and news mentions, and then use Gemini to generate a comprehensive report.

After running the cell above, you should see output in the console that includes a `public URL`. Click on that URL to open your Streamlit app in a new tab.

Let me know once you've tried it out!